# Palm Ripeness Deployment (Raspberry Pi 4B)

Convert the best Keras checkpoint to TensorFlow Lite variants and package an inference script for Raspberry Pi 4 Model B (4GB) with Camera Module 3. Update the dataset paths below before running.

In [20]:
import os, json, datetime, pathlib
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Paths and run flags — update TRAIN_ROOT/TEST_ROOT to your dataset locations
MODEL_H5 = "saved_models/palm_ripeness_best_20260311_170850.h5"
TRAIN_ROOT = r"C:\Users\jeffy\iCloudDrive\College\Y4S1\BERN4973 Final Year Project PSM1\Assignment\Dataset\Dataset1\Train"  # TODO: update if different
TEST_ROOT  = r"C:\Users\jeffy\iCloudDrive\College\Y4S1\BERN4973 Final Year Project PSM1\Assignment\Dataset\Dataset1\Test"   # TODO: update if different
OUTPUT_DIR = "saved_models"
IMG_SIZE   = (224, 224)
DO_INT8    = False  # set True only if you have a calibration subset available

os.makedirs(OUTPUT_DIR, exist_ok=True)
TIMESTAMP = pathlib.Path(MODEL_H5).stem.replace("palm_ripeness_best_", "") or datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Using model: {MODEL_H5}")
print(f"Output dir: {OUTPUT_DIR}")

Using model: saved_models/palm_ripeness_best_20260311_170850.h5
Output dir: saved_models


In [21]:
# Detect class names from training directory (sorted)
def get_class_names(root):
    if not os.path.exists(root):
        print(f"Path not found: {root}")
        return []
    return sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])

class_names = get_class_names(TRAIN_ROOT)
num_classes = len(class_names)
print("Detected classes:", class_names)
print("Num classes:", num_classes)

Detected classes: ['Overripe', 'Ripe', 'Underripe']
Num classes: 3


In [22]:
# Load the trained Keras model (.h5)
model = keras.models.load_model(MODEL_H5)
model.summary()
print(f"Loaded model with {model.count_params():,} parameters")

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,853 (9.24 MB)

 Trainable params: 164,611 (643.01 KB)

 Non-trainable params: 2,258,240 (8.61 MB)

 Optimizer params: 2 (12.00 B)

Loaded model with 2,422,851 parameters


In [23]:
def save_tflite(tflite_bytes, name):
    path = os.path.join(OUTPUT_DIR, name)
    with open(path, "wb") as f:
        f.write(tflite_bytes)
    size_mb = len(tflite_bytes) / (1024 * 1024)
    print(f"Saved {name} ({size_mb:.2f} MB)")
    return path

# Float32 reference
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float32 = converter.convert()
float32_path = save_tflite(tflite_float32, f"palm_ripeness_best_{TIMESTAMP}_float32.tflite")

# Float16 quantization (ARM-friendly)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_float16 = converter.convert()
float16_path = save_tflite(tflite_float16, f"palm_ripeness_best_{TIMESTAMP}_float16.tflite")

# Optional: full INT8 quantization with representative dataset
int8_path = None
if DO_INT8:
    def representative_data_gen():
        ds = tf.keras.utils.image_dataset_from_directory(
            TRAIN_ROOT,
            labels=None,
            image_size=IMG_SIZE,
            batch_size=1,
            shuffle=True,
        ).take(200)  # adjust as needed
        for batch in ds:
            yield [preprocess_input(tf.cast(batch, tf.float32))]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_data_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    tflite_int8 = converter.convert()
    int8_path = save_tflite(tflite_int8, f"palm_ripeness_best_{TIMESTAMP}_int8.tflite")
else:
    print("Skipping INT8 conversion (DO_INT8=False)")

INFO:tensorflow:Assets written to: C:\Users\jeffy\AppData\Local\Temp\tmposeeivu7\assets


INFO:tensorflow:Assets written to: C:\Users\jeffy\AppData\Local\Temp\tmposeeivu7\assets


Saved artifact at 'C:\Users\jeffy\AppData\Local\Temp\tmposeeivu7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_10')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  1854669913552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669910864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669912400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669911248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669912016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669911632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669910096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669910288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669909904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669912592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  185

INFO:tensorflow:Assets written to: C:\Users\jeffy\AppData\Local\Temp\tmp373giq7z\assets


Saved artifact at 'C:\Users\jeffy\AppData\Local\Temp\tmp373giq7z'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_10')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  1854669913552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669910864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669912400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669911248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669912016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669911632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669910096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669910288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669909904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1854669912592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  185

In [24]:
# Write labels.json for inference scripts (if classes detected)
labels_path = os.path.join(OUTPUT_DIR, f"labels_{TIMESTAMP}.json")
if class_names:
    with open(labels_path, "w", encoding="utf-8") as f:
        json.dump(class_names, f, ensure_ascii=True, indent=2)
    print(f"Saved labels to {labels_path}")
else:
    print("No class names detected; labels.json not written. Set TRAIN_ROOT correctly.")

Saved labels to saved_models\labels_20260311_170850.json


In [25]:
# Quick parity check on a small test batch (if TEST_ROOT exists)
# Prefer LiteRT interpreter (ai_edge_litert) when available
try:
    from ai_edge_litert.interpreter import LiteRTInterpreter as Interpreter
    print("Using LiteRTInterpreter (ai_edge_litert)")
except ImportError:
    try:
        from tflite_runtime.interpreter import Interpreter  # fallback runtime
        print("Using tflite_runtime Interpreter")
    except ImportError:
        try:
            from tensorflow.lite.python.interpreter import Interpreter  # legacy tf-lite
            print("Using tf.lite.python Interpreter (legacy)")
        except ImportError as e:
            raise ImportError(
                "No LiteRT/tflite interpreter found. Install ai-edge-litert (preferred) or tflite-runtime."
            ) from e

def run_tflite_batch(tflite_path, batch):
    interpreter = Interpreter(model_path=tflite_path)
    # Resize input to the incoming batch shape (fixes dimension mismatch when batch>1)
    input_details = interpreter.get_input_details()[0]
    interpreter.resize_tensor_input(input_details["index"], batch.shape)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    batch_in = tf.cast(batch, input_details["dtype"])
    interpreter.set_tensor(input_details["index"], batch_in)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details["index"])
    return output

if os.path.exists(TEST_ROOT):
    test_ds = tf.keras.utils.image_dataset_from_directory(
        TEST_ROOT,
        labels="inferred",
        label_mode="categorical",
        image_size=IMG_SIZE,
        batch_size=8,
        shuffle=False,
    )
    test_batch, test_labels = next(iter(test_ds))
    test_batch_pp = preprocess_input(tf.cast(test_batch, tf.float32))

    keras_preds = model.predict(test_batch_pp, verbose=0)
    lite_preds = run_tflite_batch(float16_path, test_batch_pp)

    keras_top1 = np.argmax(keras_preds, axis=1)
    lite_top1 = np.argmax(lite_preds, axis=1)
    top1_match = np.mean(keras_top1 == lite_top1)
    print(f"Top-1 agreement (keras vs float16 tflite) on batch: {top1_match:.2%}")
else:
    print("TEST_ROOT not found; skipping parity check.")

Using tf.lite.python Interpreter (legacy)
Found 180 files belonging to 3 classes.
Top-1 agreement (keras vs float16 tflite) on batch: 100.00%


C:\Users\jeffy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [27]:
# Generate a lightweight Pi inference script
pi_script_path = os.path.join(OUTPUT_DIR, "pi_inference.py")
labels_fname = os.path.basename(labels_path) if 'labels_path' in globals() else "labels.json"

def write_pi_script():
    script = """
import argparse, json, time, os
import numpy as np
from PIL import Image

# Prefer LiteRT (ai_edge_litert); fall back to tflite_runtime, then legacy tf.lite
try:
    from ai_edge_litert.interpreter import LiteRTInterpreter as Interpreter
    print("Using LiteRTInterpreter (ai_edge_litert)")
except ImportError:
    try:
        from tflite_runtime.interpreter import Interpreter
        print("Using tflite_runtime Interpreter")
    except ImportError:
        try:
            from tensorflow.lite.python.interpreter import Interpreter
            print("Using tf.lite.python Interpreter (legacy)")
        except ImportError as e:
            raise ImportError("No LiteRT/tflite interpreter found. Install ai-edge-litert (preferred) or tflite-runtime.") from e

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE = (224, 224)


def load_labels(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_image(path):
    img = Image.open(path).convert("RGB")
    img = img.resize(IMG_SIZE)
    arr = np.array(img, dtype=np.float32)
    arr = preprocess_input(arr)  # [-1,1]
    return np.expand_dims(arr, axis=0)


def run_inference(model_path, labels_path, image_path, warmup=2, runs=5):
    interpreter = Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    labels = load_labels(labels_path)
    batch = load_image(image_path).astype(input_details['dtype'])

    # Warmup
    for _ in range(warmup):
        interpreter.set_tensor(input_details['index'], batch)
        interpreter.invoke()

    timings = []
    for _ in range(runs):
        start = time.time()
        interpreter.set_tensor(input_details['index'], batch)
        interpreter.invoke()
        out = interpreter.get_tensor(output_details['index'])
        timings.append((time.time() - start) * 1000)

    probs = out[0]
    top1 = int(np.argmax(probs))
    return {{
        "top1_index": top1,
        "top1_label": labels[top1] if top1 < len(labels) else str(top1),
        "top1_prob": float(probs[top1]),
        "avg_ms": float(np.mean(timings)),
        "runs": runs,
    }}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", default="palm_ripeness_best_{timestamp}_float16.tflite", help="Path to TFLite/LiteRT model")
    parser.add_argument("--labels", default="{labels_fname}", help="Path to labels.json")
    parser.add_argument("--image", required=True, help="Path to image for inference")
    parser.add_argument("--runs", type=int, default=5, help="Number of timed runs")
    args = parser.parse_args()

    result = run_inference(args.model, args.labels, args.image, runs=args.runs)
    print(result)


if __name__ == "__main__":
    main()
""".format(timestamp=TIMESTAMP, labels_fname=labels_fname)

    with open(pi_script_path, "w", encoding="utf-8") as f:
        f.write(script)
    print(f"Wrote Pi inference script to {pi_script_path}")

write_pi_script()

Wrote Pi inference script to saved_models\pi_inference.py


### Raspberry Pi run steps (summary)

1) Copy the chosen `.tflite` (float16 recommended) and `labels_*.json` plus `pi_inference.py` to the Pi.
2) On Pi (64-bit OS): `sudo apt update && sudo apt install python3-pip python3-pil libatlas-base-dev`; then install LiteRT wheel: `pip install --no-cache-dir ai-edge-litert==2.20.0` (or the latest for ARMv8). If it fails, fall back to `tflite-runtime` (`pip install --no-cache-dir tflite-runtime`) or `tensorflow-cpu` (heavier).
3) Test an image: `python3 pi_inference.py --model palm_ripeness_best_<timestamp>_float16.tflite --labels labels_<timestamp>.json --image /path/to/test.jpg`.
4) For Camera Module 3, wrap capture with `picamera2` and feed the frame into the same preprocessing (224x224, MobileNetV2 `preprocess_input`).
5) Benchmark on 10–50 images to choose between float16 vs int8; target ~2–4 FPS on Pi 4B CPU. LiteRT should match or slightly improve latency vs legacy tf.lite runtime.